# Unit 02 - Confounding (Demo)

**Atoms served:** `U02-A2` (confounders), `U02-A4` (more data does not fix bias), `U10-A8` (Simpson's paradox)

**Estimated runtime:** ~20 seconds in Colab

**After this notebook you can:** simulate a confounder, read Simpson's paradox in a small table, and explain why a larger sample can make a biased estimate look more trustworthy without making it correct.

## Without code

Skip every code cell. Read the printed tables and plot titles as if they were handouts.

1. In the naive comparison table, note the raw treatment gap. Then read the adjusted gap after stratifying on the confounder.
2. In the Simpson's paradox table, women have the higher admit rate in **both** departments while men have the higher rate overall. Read the application counts to see why.
3. In the precision plot, watch the point estimate stay near the same wrong value while the error bars shrink as `n` grows.

You should reach the same three conclusions as someone who ran the code.

## 1. The question

An e-commerce team added a checkout redesign (treatment) and compared conversion to the old page (control). Conversion also depends on whether traffic arrived from paid ads - a channel the redesign team did not randomise.

**Can we trust a simple difference of group averages?** And if we collect more sessions, does the bias go away?

## 2. Setup

We install the course stack, set a visible seed, and import libraries. Every number below comes from this seed.

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

We simulate 4,000 checkout sessions. Each row has a treatment indicator `D_i`, a confounder `paid_channel`, and conversion `Y_i`.

The redesign has **no true effect** (`ATE = 0`). Paid traffic converts higher **and** is over-represented in the treatment group because marketing ran the test on campaign landing pages.

In [ ]:
# True ATE is zero; paid channel lifts conversion and is imbalanced across arms
n = 4000
paid = np.random.binomial(1, 0.35, n)
D = np.random.binomial(1, 0.55 + 0.25 * paid)  # treatment skewed toward paid traffic
base = 0.12 + 0.18 * paid  # paid channel converts higher
Y = np.random.binomial(1, np.clip(base + 0.00 * D, 0, 1))

df = pd.DataFrame({'D_i': D, 'paid_channel': paid, 'Y_i': Y})
print('Sessions:', len(df))
print(df.groupby('D_i')['paid_channel'].mean().rename('share_paid_traffic'))

## 4. The naive move

A reasonable first step is the difference of group means - the same estimand the course calls `ATE` when assignment is clean.

In [ ]:
naive = df.groupby('D_i')['Y_i'].mean()
naive_ate = naive[1] - naive[0]
print('Control conversion:', round(naive[0], 4))
print('Treatment conversion:', round(naive[1], 4))
print('Naive ATE (treatment - control):', round(naive_ate, 4))

The naive comparison shows a **positive** treatment effect even though the redesign does nothing. Paid traffic is doing the work.

## 5. What actually happens

**Simpson's paradox** - Berkeley 1973 admissions, as in `V32`. Each subgroup can favour women while the overall table favours men.

In [ ]:
# Two departments, aggregate reverses (Berkeley pattern).
# Men mostly applied to the department that admits almost everyone.
simpson = pd.DataFrame({
    'department': ['A (easy)', 'A (easy)', 'B (selective)', 'B (selective)', 'Overall', 'Overall'],
    'gender': ['Men', 'Women', 'Men', 'Women', 'Men', 'Women'],
    'admitted': [85, 9, 2, 25, 87, 34],
    'applied': [100, 10, 10, 100, 110, 110]
})
simpson['rate'] = simpson['admitted'] / simpson['applied']
print(simpson.pivot(index='department', columns='gender', values='rate').round(3))
print('\nApplications by department:')
print(simpson.pivot(index='department', columns='gender', values='applied'))
print('\nWomen are admitted at a HIGHER rate in both departments,')
print('yet men have the higher overall rate. Same numbers, opposite conclusions.')

Within each department, women are admitted at a higher rate. Combined, men look favoured - because men overwhelmingly applied to the department that admits 85% of applicants, while women overwhelmingly applied to the one that admits 25%. Department is the confounder, and aggregating over it inverts the answer.

Back to our checkout simulation: stratify on `paid_channel` and the fake lift disappears.

In [ ]:
adj = df.groupby(['paid_channel', 'D_i'])['Y_i'].mean().unstack()
adj_ate = (adj.loc[1, 1] - adj.loc[1, 0] + adj.loc[0, 1] - adj.loc[0, 0]) / 2
print('Conversion by channel and arm:\n', adj.round(4))
print('Adjusted ATE (simple average of channel-specific gaps):', round(adj_ate, 4))

After adjusting for channel, the estimated effect is near zero - the truth we coded.

**More data does not fix bias (`U02-A4`).** This atom is not on video; Kohavi ch. 1 and this notebook carry it.

Watch the naive estimate stay wrong while its standard error shrinks as `n` grows.

In [ ]:
sample_sizes = [200, 500, 1000, 2000, 5000, 10000]
estimates, ses = [], []
for n_i in sample_sizes:
    paid_i = np.random.binomial(1, 0.35, n_i)
    D_i = np.random.binomial(1, 0.55 + 0.25 * paid_i)
    base_i = 0.12 + 0.18 * paid_i
    Y_i = np.random.binomial(1, np.clip(base_i, 0, 1))
    tmp = pd.DataFrame({'D_i': D_i, 'Y_i': Y_i})
    diff = tmp.groupby('D_i')['Y_i'].mean()
    ate_i = diff[1] - diff[0]
    se_i = np.sqrt(tmp[tmp.D_i==1].Y_i.var()/ (tmp.D_i==1).sum() + tmp[tmp.D_i==0].Y_i.var()/ (tmp.D_i==0).sum())
    estimates.append(ate_i)
    ses.append(se_i)

fig, ax = plt.subplots()
ax.errorbar(sample_sizes, estimates, yerr=1.96*np.array(ses), fmt='o-')
ax.axhline(0, linestyle='--')
ax.set_xlabel('sample size n')
ax.set_ylabel('naive ATE estimate')
ax.set_title('More data does not fix the bias - intervals shrink around the wrong value')
plt.show()
print('True ATE = 0. Naive estimates stay positive; SE falls like 1/sqrt(n).')

## 6. What you do about it

- **Name the confounder** before you trust a gap (`U02-A2`, `V07` coupon-field story).
- **Stratify or adjust** when you must work observationally; **randomise** when you can, because randomisation breaks confounding at assignment.
- **Do not confuse precision with correctness (`U02-A4`).** A tight confidence interval around the wrong number is still wrong.

**When this matters less:** If assignment is random and `SUTVA` holds, simple differences are the honest estimand - confounding is the observational failure mode.

---

**Takeaway:** Confounders create plausible treatment effects where none exist. Simpson's paradox shows aggregation can lie even without malice. Collecting more data makes biased estimates look more precise, not more true.

**Back to the unit:** [V1 unit 02 README](../V1/units/unit-02-causality-and-confounding/README.md) · [V2 unit 02 README](../V2/units/unit-02-causality-and-confounding/README.md)